# Does HFTD Eligibility Predict SGIP Battery Funding?

**Question:** Among residential battery storage applicants to California's Self-Generation
Incentive Program (SGIP) in PG&E territory, are applicants in a High Fire-Threat District
(HFTD Tier 2/3), who qualify for enhanced incentives, more likely to reach funded (`Paid`)
status than non-HFTD applicants in the same zip code who applied at the same time?

**Data:**
- SGIP Weekly Statewide Report: application-level data, downloaded from the SGIP website
- U.S. Census Bureau ACS 5-year estimates (2020–2024): housing units and median household
  income by ZIP Code Tabulation Area (ZCTA), from the Census Data API

See `plan.md` for the full analysis plan.

## 1. Setup

Imports and constants. The Census API key is read from a `.env` file (excluded from the repo
by `.gitignore`). To rerun this notebook, create a `.env` file next to it containing:

```
CENSUS_API_KEY=your_key_here
```

A free key can be requested at https://api.census.gov/data/key_signup.html.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# Load variables from .env into the environment, then read the Census key
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
if not CENSUS_API_KEY:
    raise RuntimeError("CENSUS_API_KEY not found. Add it to a .env file (see above).")

# Folder for downloaded raw files (gitignored)
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# SGIP report URL: always serves the latest weekly snapshot as an .xlsx file
SGIP_REPORT_URL = "https://www.selfgenca.com/documents/reports/statewide_projects"

# ACS 5-year vintage (2024 = 2020-2024 estimates) and the variables to pull,
# mapped from Census variable codes to readable column names
ACS_YEAR = 2024
ACS_VARIABLES = {
    "B25001_001E": "housing_units",            # Total housing units
    "B19013_001E": "median_household_income",  # Median household income (2024 dollars)
}

## 2. Pull the SGIP application data

SGIP does not offer a formal API, but the report URL returns the latest weekly `.xlsx`
directly, so the notebook downloads it with `requests` and no manual download is needed.

The report is replaced every week and application statuses keep changing, so the file is
saved under its original dated filename and the snapshot date is printed. Results are tied to
that snapshot. If a file with the same name is already in `data/`, the download is skipped.

In [2]:
def download_sgip_report(url, data_dir):
    """Download the current SGIP weekly report and return its local path.

    Parameters
    ----------
    url : str
        SGIP report URL that serves the latest .xlsx snapshot.
    data_dir : Path
        Folder where the file is saved.
    """
    # stream=True: read the headers (to get the dated filename) before downloading the body;
    # timeout=120: seconds to wait for the server before giving up
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        # Header looks like: attachment; filename="Weekly Statewide Report_09_27_2026(posted 09_27_26).xlsx"
        disposition = response.headers["Content-Disposition"]
        filename = re.search(r'filename="(.+?)"', disposition).group(1)
        local_path = data_dir / filename

        if local_path.exists():
            print(f"Already downloaded: {filename}")
            return local_path

        with open(local_path, "wb") as file:
            # chunk_size=1 MB: write the ~25 MB file in pieces rather than holding it in memory
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                file.write(chunk)
    print(f"Downloaded: {filename}")
    return local_path


sgip_path = download_sgip_report(SGIP_REPORT_URL, DATA_DIR)

# Snapshot date is the first MM_DD_YYYY in the filename
snapshot_date = pd.to_datetime(
    re.search(r"\d{2}_\d{2}_\d{4}", sgip_path.name).group(), format="%m_%d_%Y"
)
print(f"SGIP snapshot date: {snapshot_date:%B %d, %Y}")

Downloaded: Weekly Statewide Report_09_27_2026(posted 09_27_26).xlsx
SGIP snapshot date: September 27, 2026


The data is on the first sheet. Rows 1–3 are a title, a date stamp and a blank row, so the
column headers are on Excel row 4 (`header=3`, counting from zero).

In [3]:
# sheet_name=0: first sheet (the data; the second sheet is a field dictionary)
# header=3: column names are on Excel row 4
sgip_raw = pd.read_excel(sgip_path, sheet_name=0, header=3)

print(f"{len(sgip_raw):,} rows, {sgip_raw.shape[1]} columns")
sgip_raw.head()

106,585 rows, 51 columns


,SGIP Administrator,Application Code,Program Year,Step,Budget Category,Fully Qualified State,Equipment Type,Rated Capacity [kW],Energy Storage Capacity (kWh),Fuel Type,...,Cancelled Date,CSI Rating,Advance Payment Program (AB 209),Total Eligible Project Cost - Solar,Total Eligible Project Cost - Storage,Solar Array Manufacturers,Solar Inverter Manufacturers,Funding Source,"Located in HFTD (Tier 2, Tier 3)",Experienced Two PSPS Events (True/False)
0,Southern California Edison,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,750.0,1500.0,NaN,...,05/04/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
1,Southern California Edison,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,125.0,250.0,NaN,...,05/04/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
2,Center for Sustainable Energy,NaN,NaN,NaN,NaN,Cancelled,Electrochemical Storage,261.0,522.0,NaN,...,06/01/17,NaN,NaN,NaN,NaN,NaN,NaN,Ratepayer,NaN,NaN
3,LA Dept. of Water and Power,LA-SGIP-2025-0001,2025.0,6.0,Residential Solar and Storage Equity - AB 209,RRF Technical Review,Electrochemical Storage,23.0,26.4,NaN,...,NaN,11.050649,Yes,48938.59,41485.71,Mission Solar Energy LLC,Tesla Inc.,State,Tier 2,Not Applicable
4,LA Dept. of Water and Power,LA-SGIP-2025-0002,2025.0,6.0,Residential Solar and Storage Equity - AB 209,RRF Suspended,Electrochemical Storage,34.5,39.6,NaN,...,NaN,23.566935,Yes,104367.86,47142.86,Tesla Inc.,Tesla Inc.,State,Not Applicable,Not Applicable


## 3. Pull Census ACS data by ZCTA

The Census Data API returns JSON: a list of rows whose first row is the column headers.
Since the 2020 ACS, ZCTAs cannot be filtered by state in the request, so all ZCTAs
nationwide are pulled here. The data is narrowed to California in the cleaning step.

In [4]:
def fetch_acs_zcta(variables, year, api_key):
    """Pull ACS 5-year estimates for every ZCTA and return them as a DataFrame.

    Parameters
    ----------
    variables : dict
        Census variable codes mapped to the column names to use.
    year : int
        ACS 5-year vintage (e.g. 2024 = 2020-2024 estimates).
    api_key : str
        Census Data API key.
    """
    url = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        "get": ",".join(variables),                # variables to return
        "for": "zip code tabulation area:*",       # geography: all ZCTAs
        "key": api_key,
    }
    # timeout=120: seconds to wait for the server before giving up
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()

    header, *rows = response.json()
    acs = pd.DataFrame(rows, columns=header)
    return acs.rename(columns={**variables, "zip code tabulation area": "zcta"})


acs_raw = fetch_acs_zcta(ACS_VARIABLES, ACS_YEAR, CENSUS_API_KEY)

print(f"{len(acs_raw):,} ZCTAs")
acs_raw.head()

33,772 ZCTAs


,housing_units,median_household_income,zcta
0,7570,19454,00601
1,17610,21420,00602
2,26239,20933,00603
3,2681,20992,00606
4,12636,24496,00610
